# Train Card Strength Model

**Objectives:**
- Prepare the dataset for training by splitting and scaling features.
- Train an late fusion model.
- Evaluate model accuracy using MAE and RMSE metrics.

In [109]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

In [ ]:
INPUT_DATA_PATH = "../data/training/card_strength_input.parquet"
LABEL_DATA_PATH = "../data/training/card_strength_metric.xlsx"
OUTPUT_MODEL_PATH = "../models/card_strength_model.pth"
SEED = 1893

In [ ]:
np.random.seed(SEED)
df_input = pd.read_parquet(INPUT_DATA_PATH)
df_label = pd.read_excel(LABEL_DATA_PATH)

# TODO: merge data
df = df_input.merge(df_label, "left", "oracle_id")
print(f"Shape: {df.shape}")
df.head()

## Data Filtering

Exclude most recent set (and every card that has been release since to avoid inaccuracies)

In [ ]:
# TODO: from all sets find the most recent (that has been released)
# -> filter all cards from that set and cards that are released after that (like secret lairs)

## Feature Preparation

In [ ]:
embedding_cols = [col for col in df.columns if col.startswith('embed_')]
embedding_features = df[embedding_cols].values
boolean_cols = [col for col in df.columns if col.startswith('bool_')]
boolean_features = df[boolean_cols].values
numerical_cols = [col for col in df.columns if col.startswith('num_')]
numerical_features = df[numerical_cols].values

X = np.hstack((embedding_features, numerical_features, boolean_features))
y = df['log_power_score'].values
print(f"Feature Matrix shape: {X.shape}")

## Correlation Analysis

- Calculate Pearson correlation between features and the target power score.
- Plot the top positive and negative correlations to validate feature engineering.

In [113]:
feature_names = embedding_cols + numerical_cols + boolean_cols
correlations = df[feature_names + ['log10_power_score']].corr()['log10_power_score'].drop('log10_power_score')
top_correlations = pd.concat([correlations.nsmallest(10), correlations.nlargest(10)]).sort_values()

plt.figure(figsize=(12, 8))
sns.barplot(x=top_correlations.values, y=top_correlations.index)
plt.title('Top Correlations with Power Score', fontsize=16)
plt.show()

## Data Splitting

- Split data into Training (60%), Validation (20%), and Test (20%) sets.
- Ensure stratification and reproducibility using the defined seed.

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=SEED)
print(f"Training data: {X_train.shape[0]} cards")

## Train late fusion model

In [ ]:
def train_model(data_path="data/card_strength_dataset.parquet", epochs=50, batch_size=64):
    print(f"Loading data from {data_path}...")
    df = pd.read_parquet(data_path)
    
    # Identify columns
    embedding_cols = [col for col in df.columns if col.startswith('embed_')]
    numerical_cols = [col for col in df.columns if col.startswith('num_')]
    boolean_cols = [col for col in df.columns if col.startswith('bool_')]
    target_col = 'log10_power_score'
    
    # 1. Prepare Features
    text_embeddings = df[embedding_cols].values
    
    # Scale numerical features (fill NaN with -1 as MTG values are mostly >=0)
    tabular_features = df[numerical_cols + boolean_cols].fillna(-1).values
    scaler = StandardScaler()
    tabular_features = scaler.fit_transform(tabular_features)
    
    targets = df[target_col].values
    
    # 2. Split Data
    X_text_train, X_text_test, X_tab_train, X_tab_test, y_train, y_test = train_test_split(
        text_embeddings, tabular_features, targets, test_size=0.2, random_state=1893
    )
    
    # 3. Create DataLoaders
    train_ds = MTGDataset(X_text_train, X_tab_train, y_train)
    test_ds = MTGDataset(X_text_test, X_tab_test, y_test)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    
    # 4. Initialize Model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    model = MTGFusionModel(
        text_dim=len(embedding_cols), 
        tabular_dim=tabular_features.shape[1]
    ).to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
    
    # 5. Training Loop
    best_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for texts, tabs, labels in train_loader:
            texts, tabs, labels = texts.to(device), tabs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(texts, tabs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * texts.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for texts, tabs, labels in test_loader:
                texts, tabs, labels = texts.to(device), tabs.to(device), labels.to(device)
                outputs = model(texts, tabs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * texts.size(0)
        
        val_loss /= len(test_loader.dataset)
        scheduler.step(val_loss)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), "best_fusion_model.pth")
            
    print(f"Training complete. Best Val Loss: {best_loss:.4f}")